# BINF TP7 - Analyse d'expression d'ARN de cellules individuelles

Dans ce TP nous allons utiliser la bibliothèque scanpy pour manipuler des données de séquençage d'ARN de cellules uniques.

La documentation de scanpy est disponnible ici : https://scanpy.readthedocs.io/en/latest/tutorials/index.html.

Les données que nous alons utiliser sont disponnibles ici : https://www.dropbox.com/scl/fo/id39ua0vybeqep20nqny0/AC9xDkvQYH7s76rWpjMn2xQ?rlkey=zd6d31k5g2qasxxmo1wuh7dcz&st=jeulzo8k&dl=0



In [ ]:
!pip install scanpy
!pip install leidenalg

In [ ]:
import scanpy as sc

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

# Exercice 1 : chargement des données et analyse préliminaire

Q1. Uploadez les données dans votre gdrive et chargez les en utilisant la fonction *read_10x_mtx*.

In [ ]:
adata = sc.read_10x_mtx(
   "/content/sample_data/hg19",  # the directory with the `.mtx` file
    cache=True)

Q2. Combien de gènes et cellules sont présents dans les données ?

In [ ]:
print(adata)

Q3. Affichez une heatmap de la matrice totale. Utilisez une échelle logarithmique pour aider à séparer les différentes valeurs.

In [ ]:
from matplotlib import pyplot as plt

plt.figure()
plt.imshow(adata.X.toarray())

Q4. Que voyez-vous ?

On ne voit rien!! Il y a trop de lignes et de colonnes.

Q5. Affichez une version réduite aux 1000 premiers gènes et cellules.

In [ ]:
import numpy as np

plt.figure()
a = np.log1p(adata.X.toarray())
plt.imshow(a[:1000,:1000])

Q6. Qu'observez-vous maintenant ?

On peut voir des différences avec des colonnes qui se démarquent.

Q7. Affichez la distribution du nombre de gènes par cellule.

In [ ]:
gene_per_cell = np.sum(adata.X > 0, axis=1)
plt.figure()
plt.hist(gene_per_cell, bins=50)
plt.xlabel("Nombre de gènes par cellule")
plt.ylabel("Compte")

Q8. Affichez la distribution du nombre de comptes par cellule.

In [ ]:
cnt_per_cell = np.sum(adata.X, axis=1)
plt.figure()
plt.hist(cnt_per_cell, bins=50)
plt.xlabel("Nombre de comptes par cellule")
plt.ylabel("Compte")

Q9. Affichez la distribution du nombre de cellules par gène.

In [ ]:
plt.figure()
plt.hist(np.sum(adata.X > 0, axis=0).A1, bins=50)
plt.xlabel("Nombre de cellules par gène")
plt.ylabel("Compte")

Q10. Donnez la liste des 20 gènes les plus exprimés.

In [ ]:
expr = np.sum(adata.X, axis=0).A1
top_idx = np.argsort(expr)[::-1][:20]
for ti in top_idx:
  print(adata.var_names[ti], expr[ti])

# Exercice 2 : autres mesures de qualité

Q11. Combien de gènes proviennent des mitochondries ? (Ils ont un nom commençant par “MT-”)

In [ ]:
mt_genes = adata.var_names.str.startswith('MT-')
print(sum(mt_genes), list(adata.var_names[mt_genes]))

Q12. Affichez le pourcentage de gènes de mitochondrie en fonction compte de gène par cellule.

In [ ]:
mt_genes_pct_per_cell = adata.X[:, mt_genes].sum(axis=1) / adata.X.sum(axis=1).A * 100
print(mt_genes_pct_per_cell)

plt.figure()
plt.plot(cnt_per_cell, mt_genes_pct_per_cell, 'x')
plt.xlabel('Cnt per cell')
plt.ylabel('Mito gene %')

Q13. Qu'observez-vous ? Où sont les outliers ?

La majorité des cellules ont < 10% de gènes mito, même celles avec un faible compte. Les outliers seront dont > ~10% gènes mito.

Q14. Affichez le nombre de gènes en fonction du compte par cellule.

In [ ]:
plt.figure()
plt.plot(cnt_per_cell, gene_per_cell, 'x')
plt.xlabel('Cnt per cell')
plt.ylabel('Gene per cell')

Q15. Qu'observez-vous ? Où sont les outliers ?

On a une répartition linéaire assez propre. On ne distingue pas vraiment d'outliers.

Q16. Ajoutez à la structure de données le compte de gènes par celluler nommé“total_counts” et le pourcentage de comptes de mito par cellule nommé “pct_counts_mt”.


In [ ]:
adata.obs["pct_counts_mt"] = mt_genes_pct_per_cell

# Exercice 3 : filtrage des données

Q17. Enlevez de la structure les gènes présents dans < 3 cellules et les cellules avec < 200 comptes

In [ ]:
adata = adata[np.sum(adata.X, axis=1).A1 >= 200, :]
adata = adata[:, np.sum(adata.X > 0, axis=0) >= 3]

Q18. Enlevez les cellules avec un trop gros pourcentage de comptes gènes de mitochondries - déterminez vous-même le seuil adéquat.


In [ ]:
adata = adata[mt_genes_pct_per_cell < 10, :]

Q19. Enlever les cellules avec un trop gros nombre de gènes - déterminez vous-même le seuil adéquat.

In [ ]:
gene_per_cell = np.sum(adata.X > 0, axis=1)
adata = adata[gene_per_cell < 2500, :]

In [ ]:
print(adata)

#Exercice 4 : normalisation

Différentes cellules expriment différentes quantité d'ARN ce qui peut biaiser les comparaisons dans les étapes futures. Une étape importante de l'analyse est de normaliser la quantité d'ARN entre cellules.

Q20. Normalisez la matrice de données en divisant les comtes de gènes pour chaque cellule par son compte d'ARN total divisé par la médiane des comptes total divisé par un facteur cpm=1e4.

In [ ]:
from sklearn.utils import sparsefuncs

cpm = 1e4
cnt_per_cell = np.sum(adata.X, axis=1)
norm_factor_per_cell = cnt_per_cell / np.median(cnt_per_cell, axis=0)[0] / cpm
X = adata.X
sparsefuncs.inplace_row_scale(X, 1.0 / norm_factor_per_cell.A1)
adata.X = X

Q21.  Appliquer une transformé log(1+X) sur la matrice.

In [ ]:
adata.X = adata.X.log1p()

Q22. Pourquoi log(1+X) et pas simplement log(X) ?

Pour éviter les problèmes avec les données à 0 ($\log(0) = -\infty$).

# Exercice 5 : gènes très variables

Q23. Détectez et affichez les gènes très variables en utilisant les fonctions *highly_variable_genes* (il y en a 2 avec le même nom dans 2 sous-modules différents).


In [ ]:
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
sc.pl.highly_variable_genes(adata)

Avant d'aller plus loin, sauvegardons l'état actuel de la matrice dans l'attribut raw:

In [ ]:
adata.raw = adata

Q24. Enlever les gènes qui ne sont **PAS** très variables.

In [ ]:
adata = adata[:, adata.var.highly_variable]

Q25. Pourquoi ceux-ci ?

Les gènes qui ne varient pas dans le jeu de données ne contiennent pas d'information.

# Exercice 6: scaling

Q26. Nodifiez la matrice pour centrer et normaliser le niveau d'expression tel que chaque gène ait une moyenne de 0 et une variance de 1.


In [ ]:
from scipy.sparse import csr_matrix

tmp = adata.X.toarray()
adata.X = csr_matrix((tmp - tmp.mean(axis=0)) / np.sqrt(tmp.var(axis=0)))

# Exercice 6 : visualisation PCA


Q27. Utilisez *sc.tl.pca* pour lancer une PCA sur la matrice.

In [ ]:
sc.tl.pca(adata, svd_solver='arpack')

Q28. Utilisez *sc.pl.pca* pour visualiser le résultat.


In [ ]:
sc.pl.pca(adata, color='CST3')

Q29. Qu'obervez-vous ?

On ne voit pas grand chose.


Q30. Utilisez *sc.pl.pca_variance_ratio* pour afficher la variance expliquée pour les PC successives.

In [ ]:
sc.pl.pca_variance_ratio(adata, log=False)

Q31. Combien de dimensions devraient-on utiliser pour expliquer assez fidèlement ce jeu de données ?

Au moins 3, au mieux 5, les dimensions > 5 ne sont pas très informatives.



Q32. En quoi celà est-il un problème pour la représentation ?

On ne peut pas représenter des données > 3 dimensions.

# Exercice 7 : projection par umap

Q33. Utilisez *sc.tl.umap* pour calculer l'algorithme umap et *sc.pl.umap *pour visualiser les résultats.

In [ ]:
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)
sc.tl.umap(adata)

In [ ]:
sc.pl.umap(adata, color=['CST3', 'NKG7', 'PPBP'])

Q34. Quelle est la différence par rapport à la PCA ?

UMAP cherche à séparer les données pour visualisation la PCA cherche à minimiser la dimmensionalité des données.

Q35. Expliquez brièvement comment fonctionne l'algorithme UMAP?

UMAP cherche à projeter les données dans un nombre fixe de dimensions de tel sorte que les distances entre points dans l'espace à haute dimension de départ soient conservées.

# Exercice 8 : expression différentielle

Q36. Utiliser *sc.tl.leiden* pour calculer les cluster dans les données projetées.

In [ ]:
sc.tl.leiden(adata)

Q37. Expliquez brièvement comment fonctionne cet algorithme.

C'est un algorithm qui cherche les communautées dans un graphe. Celà va nous permettre de détecter les cluster dans les données projetées.

Q38. Affichez le résultat avec *sc.pl.umap(adata, color=['leiden'])*


In [ ]:
sc.pl.umap(adata, color=['leiden'])

Q39. Utilisez les cluster définis à l'étape précédente pour analyser les gènes exprimés différentiellement. On fera ça en utilisant *sc.tl.rank_genes_groups* avec “*method='wilcoxon'*”.


In [ ]:
sc.tl.rank_genes_groups(adata, 'leiden', method='wilcoxon')

Q40. Expliquez brièvement ce que fait cette fonction.

Cette méthode cherche les gènes représentatifs de chaque cluster. C'est à dire ceux qui sont exprimés significativement différement des autres clusters.

Q41. Visualisez les résultats avec *sc.pl.rank_genes_groups*, vous pouvez limiter à e.g. 25 gènes.


In [ ]:
sc.pl.rank_genes_groups(adata, n_genes=25, sharey=False)

Q42. Assignez un type de cellule à chaque cluster Voici une liste de gènes exprimés en particulier dans certains types de cellules :

In [ ]:
marker_genes = ['IL7R', 'CD79A', 'MS4A1', 'CD8A', 'CD8B', 'LYZ', 'CD14',
                'LGALS3', 'S100A8', 'GNLY', 'NKG7', 'KLRB1',
                'FCGR3A', 'MS4A7', 'FCER1A', 'CST3', 'PPBP']

Utilisez cette liste et l'expression différentielle de l'étape précédente pour trouver à quel type de cellule correspond chaque cluster.

In [ ]:
sc.pl.dotplot(adata, marker_genes, groupby="leiden")